# Indicator 5: Fibonacci Retracement

Fibonacci levels are computed from a rolling high/low window, generating horizontal
support & resistance levels at 23.6%, 38.2%, 50%, 61.8%, and 78.6% retracements.

| Condition | Signal |
|-----------|--------|
| Price falls to 61.8% or 50% level after a prior **uptrend** | +1 (near support) |
| Price rallies to 38.2% level after a **downtrend** and stalls | −1 (near resistance) |
| Elsewhere | 0 (neutral zone) |

**Stall confirmation (short only):** the current bar must not be making a new high
relative to the previous bar — i.e. the rally into resistance has stopped.

Steps:
  1. Over the last `window` bars, find the swing high and swing low.
  2. Determine dominant direction: whichever extreme formed last.
  3. Compute all five Fibonacci retracement levels.
  4. Signal logic (asymmetric by design):
       Uptrend   -> long  (+1) if price is near the 61.8% or 50% level
                    (deep retracements that historically attract buyers)
       Downtrend -> short (-1) if price is near the 38.2% level
                    AND the current bar is not making a new high vs the
                    prior bar (stall confirmation)

In [6]:
# ---- Indicator #: Fibonacci Retracement ------------------------------------
FIBO_LEVELS = [0.236, 0.382, 0.500, 0.618, 0.786]

LONG_LEVELS  = {0.618, 0.500}   # support zones that trigger long in an uptrend
SHORT_LEVELS = {0.382}          # resistance zone that triggers short in a downtrend


def _fib_retracement_levels(swing_low, swing_high):
    '''Return a dict {ratio: price_level} for all five Fibonacci levels.
    Levels are measured from swing_low up toward swing_high.
    A 0.382 level means price has retraced 38.2% of the full move.
    '''
    move = swing_high - swing_low
    return {r: swing_high - r * move for r in FIBO_LEVELS}


def indicator_5_fibonacci(prices_series, date_idx, window=60, proximity_pct=0.015):
    '''Fibonacci Retracement signal.

    Parameters
    ----------
    prices_series  : list of floats — full price history up to current bar
    date_idx       : int            — current bar index (0-based)
    window         : int            — lookback for swing high / low (default 60)
    proximity_pct  : float          — proximity tolerance as a fraction of
                                      current price (default 1.5%)

    Returns
    -------
     1  Price has pulled back to the 61.8% or 50% level in an uptrend (support)
    -1  Price has rallied to the 38.2% level in a downtrend and stalled (resistance)
     0  No actionable Fibonacci level nearby, or insufficient data
    '''
    if date_idx < window + 1:   # need at least one prior bar for stall check
        return 0

    window_prices = prices_series[date_idx - window + 1:date_idx + 1]
    current_price = prices_series[date_idx]
    prev_price    = prices_series[date_idx - 1]

    swing_high = max(window_prices)
    swing_low  = min(window_prices)

    if swing_high - swing_low < 1e-9:
        return 0

    high_idx = window_prices.index(swing_high)
    low_idx  = window_prices.index(swing_low)

    uptrend = high_idx > low_idx   # swing high formed after swing low

    levels    = _fib_retracement_levels(swing_low, swing_high)
    tolerance = current_price * proximity_pct

    if uptrend:
        # Long signal: price pulls back to deep support (61.8% or 50%)
        for ratio in LONG_LEVELS:
            if abs(current_price - levels[ratio]) <= tolerance:
                return 1

    else:
        # Short signal: price rallies into resistance (38.2%) and stalls
        # Stall = current bar is NOT making a new high vs the previous bar
        stalled = current_price <= prev_price
        for ratio in SHORT_LEVELS:
            if abs(current_price - levels[ratio]) <= tolerance and stalled:
                return -1

    return 0